# VRAM sweep + 학습 probe 재현 노트북

**목적 두 가지 (둘 다 중요):**

1. **다른 4대 5060 Ti에서 같은 표 재현** — 09/17 세션에서 측정한 해상도·crop VRAM sweep과
   학습 probe(9B NF4 QLoRA)를 그대로 다시 돌려 결과가 이 PC(`<호스트명>`)와 같은지 본다.
2. **PC 셧다운 재현 여부 확인 (더 급함)** — 측정에 쓴 PC가 학습 부하에서 전원이 통째로
   꺼지는 현상이 있었다. 이 PC만의 문제인지 교육장 PC 공통 문제인지 아직 모른다.
   그래서 이 노트북은 **중간에 죽어도 그 직전까지 기록이 남도록** 설계했다.

**측정에 쓴 PC는 이 노트북을 돌리는 도중 실제로 두 번 전원이 꺼졌다.** 아직 이 PC
개별 문제인지 카드 공통 문제인지 모른다. **가능하면 이 노트북보다
`gpu_capability_probe.ipynb`를 먼저(적어도 그 Stage 1까지는) 돌려보길 권한다** —
9B 모델 없이 5분 안에 이 카드가 지속 부하를 버티는지만 먼저 확인해준다. 그게
불안하면 이 노트북은 건너뛰어도 된다 — Stage 1 결과만으로도 팀에 중요한 정보다.

## 실행 방법

1. 이 저장소 루트(`AI_2_CHALLENGE/`)에서 `baseline` 가상환경으로 Jupyter를 연다.
2. 위에서 아래로 순서대로 실행한다. 셀 순서를 바꾸지 않는다 — **가벼운 설정부터
   무거운 설정 순**으로 배치했다(셧다운이 나면 어느 부하 지점에서 나는지가 로그에 남는다).
3. 첫 코드 셀(transformers 설치)은 시간이 좀 걸린다(git 커밋 고정, 최초 1회만).
4. 끝까지 돌았든 중간에 죽었든, 같은 폴더에 두 CSV가 남는다:
   - `power_log_<hostname>_<시작시각>.csv` — 1초 간격 GPU 전력·온도·사용률·VRAM (죽기 직전까지)
   - `results_<hostname>.csv` — 설정별 측정 결과 한 줄씩

**끝나면 (또는 셧다운으로 중간에 멈췄으면) 이 두 CSV 파일을 그대로 보내달라.** 특히
`power_log_*.csv`는 노트북이 끝까지 못 돌았어도 그 자체가 중요한 증거다 — 어느 시점에
전력이 어떻게 움직이다 끊겼는지가 셧다운 원인 진단의 핵심 데이터다.

## 이 노트북이 하는 것

가벼운 것부터: 추론 Full 1024 → 1536 → 2048 → crop sweep(4장@768px, 4장@512px, 9장@512px)
→ 마지막에 학습 probe(9B NF4 QLoRA, 3 step, grad_checkpointing on/off, crop 학습 1개).
각 설정은 `python -m src.run smoke/train --synthetic ...`를 별도 프로세스로 실행한다 —
설정 사이에 프로세스가 완전히 새로 뜨기 때문에 이전 설정의 VRAM 파편화가 다음 설정에
영향을 주지 않고, 한 설정이 OOM이나 크래시로 죽어도 노트북 자체(그리고 전력 로거)는
계속 산다.


## 0. transformers 버전 고정 설치

아직 이 PC에 안 깔려 있으면 실행한다. 이미 깔려 있으면 그대로 통과한다(재설치 안 함).

In [ ]:
import subprocess
import sys

PINNED_COMMIT = "ffddd25146e4225b97a6b5e5b66dd9a823bf39d1"

def _installed_transformers_commit():
    try:
        import transformers
    except ImportError:
        return None
    # pip freeze 출처 문자열에서 커밋 해시를 뽑는다. 일반 PyPI 설치면 None을 돌려준다.
    try:
        out = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            capture_output=True, text=True, check=True, timeout=60,
        ).stdout
        for line in out.splitlines():
            if line.startswith("transformers @ git+"):
                return line.strip().rsplit("@", 1)[-1]
    except Exception as exc:
        print("pip freeze 확인 실패:", exc)
    return None

current = _installed_transformers_commit()
if current == PINNED_COMMIT:
    print(f"transformers 이미 고정 커밋 설치됨: {current}")
else:
    print(f"현재: {current!r} != 고정 커밋. 설치한다 (git 클론 + wheel 빌드, 1~2분)")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         f"git+https://github.com/huggingface/transformers.git@{PINNED_COMMIT}"],
        check=True,
    )
    print("설치 완료. 커널을 재시작한 뒤 이 셀부터 다시 실행해서 버전을 확인하라.")


## 0-1. flash-linear-attention / triton 설치 시도 (선택, 시간 제한)

**왜 필요한가.** Qwen3.5-9B는 32층 중 24층이 Gated DeltaNet이다. `flash-linear-attention`
(fla)과 `triton`이 없으면 transformers가 그 24층을 pure PyTorch reference 구현으로
돌린다 — 09/17 로그에 뜬 `falling back to its reference PyTorch implementation...
much slower` 경고가 그 증거다. 이게 없으면 **지금까지의 모든 sec/step 수치가
fallback 기준**이라 실제보다 느리게 측정된 것일 수 있다.

**한 번씩만 시도한다.** 이 카드(RTX 5060 Ti, Blackwell/sm_120)는 신생 아키텍처라
prebuilt wheel이 없을 수 있다. 실패해도 여기서 멈추지 않고 pure PyTorch로 계속
진행한다 — 소스 빌드나 장시간 컴파일은 하지 않는다.


In [ ]:
def _try_install(pip_args, label):
    print(f"--- {label} 설치 시도 ---")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", *pip_args],
                       check=True, timeout=120, capture_output=True, text=True)
        print(f"{label}: 설치 성공")
        return True
    except subprocess.CalledProcessError as exc:
        print(f"{label}: 설치 실패 (exit={exc.returncode}) — 계속 진행한다")
        print((exc.stderr or "")[-1000:])
        return False
    except subprocess.TimeoutExpired:
        print(f"{label}: 시간 초과(120s) — 계속 진행한다")
        return False

try:
    import fla  # noqa: F401
    print("fla 이미 설치됨")
except ImportError:
    _try_install(["flash-linear-attention"], "flash-linear-attention")

try:
    import triton  # noqa: F401
    print("triton 이미 설치됨")
except ImportError:
    ok = _try_install(["triton"], "triton (Linux 표준 패키지)")
    if not ok:
        _try_install(["triton-windows"], "triton-windows (Windows 전용, 한 번만 시도)")

print("\n이 셀 이후 import한 패키지는 커널 재시작 전까지 이 프로세스에 고정된다.")
print("다음 셀(환경 스냅샷)에서 실제로 잡혔는지 확인한다.")


## 1. 환경 스냅샷 — 반드시 첫 코드 실행 결과로 남긴다

GPU 이름, 드라이버 버전, 총 VRAM, **우리 프로세스를 띄우기 전의 baseline VRAM 점유량**
(다른 모니터링 프로그램이 이 카드에 물려 있으면 여기서 드러난다), 주요 패키지 버전,
그리고 **fla/triton이 실제로 커널 경로로 잡히는지**(단순 import 성공과는 다르다 —
src/resources.py의 kernel_impl_snapshot()과 같은 방식으로 확인한다).

In [ ]:
import datetime
import platform
import socket
import subprocess

HOSTNAME = socket.gethostname()
START_TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print("hostname:", HOSTNAME)
print("start_ts:", START_TS)
print("platform:", platform.platform())

def nvidia_smi_query(fields):
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=10,
    ).stdout.strip()
    return [line.split(", ") for line in out.splitlines()]

print("\n--- GPU / 드라이버 ---")
for row in nvidia_smi_query("name,driver_version,memory.total,memory.used,memory.free,power.draw,temperature.gpu"):
    print("name:", row[0])
    print("driver_version:", row[1])
    print("memory.total (MiB):", row[2])
    print("memory.used (MiB) — 우리 프로세스 실행 전 baseline:", row[3])
    print("memory.free (MiB):", row[4])
    print("power.draw (W):", row[5])
    print("temperature.gpu (C):", row[6])

print("\n--- 패키지 버전 ---")
for pkg in ["torch", "transformers", "accelerate", "peft", "bitsandbytes", "fla", "triton"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{pkg}: 설치 안 됨")

print("\n--- 실제 커널 경로는 여기서 보지 않는다 ---")
print("""fla/triton이 import는 돼도 transformers가 실제로 그 커널을 골랐는지는 별개다
(모듈을 처음 import하는 시점에 고정되고, 이 노트북 프로세스 자체는 그 모듈을 아직 안
불러왔다). 5~7절에서 각 측정을 별도 `python -m src.run` 프로세스로 돌리면 그 프로세스
안에서 src/resources.kernel_impl_snapshot()이 판정한 값이 결과 CSV의
gated_delta_rule_impl / causal_conv1d_impl 컬럼에 그대로 남는다 — 거기서 최종 확인한다.""")


## 2. 전력 로거 — 별도 스레드, 1초 간격, **매 줄 flush+fsync**

PC가 통째로 꺼져도 이 CSV의 마지막 줄까지는 디스크에 남는다(OS 버퍼에만 있으면 전원이
끊길 때 날아간다 — 그래서 매 줄마다 강제로 디스크에 내린다). 노트북이 끝날 때까지
백그라운드에서 계속 돈다.


In [ ]:
import csv
import os
import threading
import time

POWER_LOG_PATH = f"power_log_{HOSTNAME}_{START_TS}.csv"
POWER_LOG_FIELDS = ["timestamp", "power_draw_w", "temperature_c", "utilization_pct", "memory_used_mib"]

_power_log_stop = threading.Event()

def _power_log_loop():
    with open(POWER_LOG_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if f.tell() == 0:
            writer.writerow(POWER_LOG_FIELDS)
            f.flush()
            os.fsync(f.fileno())
        while not _power_log_stop.is_set():
            try:
                rows = nvidia_smi_query("power.draw,temperature.gpu,utilization.gpu,memory.used")
                ts = datetime.datetime.now().isoformat()
                for row in rows:
                    writer.writerow([ts, *row])
                f.flush()
                os.fsync(f.fileno())
            except Exception as exc:
                # 폴링 자체가 실패해도 로거 스레드는 죽지 않는다 — 다음 tick에 재시도.
                print("power log 폴링 실패(계속 시도함):", exc)
            _power_log_stop.wait(1.0)

_power_log_thread = threading.Thread(target=_power_log_loop, daemon=True)
_power_log_thread.start()
print(f"전력 로거 시작: {POWER_LOG_PATH} (1초 간격, 매 줄 flush+fsync)")


## 3. 결과 CSV 준비

컬럼을 고정한다 — 5대 결과를 나중에 세로로 이어붙여 비교하기 위해서다. 한 설정이
끝날 때마다 한 줄씩 추가하고 그 자리에서 flush+fsync한다(OOM이나 크래시가 나도
그 전까지의 행은 남는다).


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RESULTS_PATH = f"results_{HOSTNAME}.csv"
RESULTS_FIELDS = [
    "hostname", "gpu_name", "mode", "long_side", "crops", "crop_long_side",
    "visual_tokens", "peak_vram_torch", "peak_vram_smi",
    "sec_per_step_steady", "sec_per_step_warmup", "status",
    "fla_version", "triton_version", "gated_delta_rule_impl", "causal_conv1d_impl",
]

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print("REPO_ROOT:", REPO_ROOT)

GPU_NAME = nvidia_smi_query("name")[0][0]

def _write_result_row(row: dict) -> None:
    is_new = not Path(RESULTS_PATH).exists()
    with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in RESULTS_FIELDS})
        f.flush()
        os.fsync(f.fileno())

print(f"결과 CSV: {RESULTS_PATH}")


## 4. 측정 실행 함수

각 설정을 별도 `python -m src.run` 프로세스로 돌린다. OOM이든 다른 에러든 여기서
잡아서 결과 CSV에 `status`로 남기고 다음 설정으로 넘어간다 — **하나가 터져도
노트북 전체가 멈추지 않는다.**


In [ ]:
def _run_subprocess(args, timeout_sec=900):
    """python -m src.run ... 을 서브프로세스로 돌리고 (returncode, stdout+stderr)를 돌려준다.

    타임아웃이 걸리면 status='TIMEOUT'으로 취급한다(진짜 셧다운이면 커널 자체가 죽어
    이 함수가 리턴하지 않는다 — 그 경우의 증거는 전력 로거 CSV가 담당한다).
    """
    proc = subprocess.run(
        [sys.executable, "-m", "src.run", *args],
        cwd=str(REPO_ROOT), capture_output=True, text=True, timeout=timeout_sec,
    )
    return proc.returncode, (proc.stdout or "") + (proc.stderr or "")


def _classify_status(returncode: int, output: str) -> str:
    if returncode == 0:
        return "ok"
    if "OutOfMemoryError" in output or "CUDA out of memory" in output:
        return "OOM"
    return f"ERROR(exit={returncode})"


def _apply_kernel_impl(row: dict, kernel_impl: dict | None) -> None:
    if not kernel_impl:
        return
    row["fla_version"] = kernel_impl.get("fla_version")
    row["triton_version"] = kernel_impl.get("triton_version")
    row["gated_delta_rule_impl"] = kernel_impl.get("gated_delta_rule_impl")
    row["causal_conv1d_impl"] = kernel_impl.get("causal_conv1d_impl")


def measure_infer(name: str, long_side=None, crops=0, crop_long_side=None, n=4):
    """smoke --synthetic 하나를 돌리고 결과 CSV에 한 줄 기록한다."""
    args = ["smoke", "-c", "9b_base_1024", "--synthetic", "-n", str(n)]
    if long_side is not None:
        args += ["--set", f"input.long_side={long_side}"]
    args += ["--set", f"input.crops={crops}"]
    if crop_long_side is not None:
        args += ["--set", f"input.crop_long_side={crop_long_side}"]

    row = {
        "hostname": HOSTNAME, "gpu_name": GPU_NAME, "mode": "infer",
        "long_side": long_side, "crops": crops, "crop_long_side": crop_long_side,
    }
    print(f"=== [infer] {name} ===")
    try:
        rc, output = _run_subprocess(args)
        status = _classify_status(rc, output)
        run_dir = REPO_ROOT / "runs" / "smoke_9b_base_1024"
        summary_path = run_dir / "summary.json"
        if status == "ok" and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            row["visual_tokens"] = summary.get("visual_tokens")
            row["peak_vram_torch"] = summary.get("peak_vram_gb")
            _apply_kernel_impl(row, summary.get("kernel_impl"))
        row["status"] = status
        print(f"  -> {status}  peak_vram_torch={row.get('peak_vram_torch')}")
        if status != "ok":
            print(output[-2000:])
    except subprocess.TimeoutExpired:
        row["status"] = "TIMEOUT"
        print("  -> TIMEOUT")
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        print("  -> ERROR:", exc)
    finally:
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
    _write_result_row(row)
    return row


def measure_train(name: str, n_steps=12, grad_checkpointing=True,
                   crops=0, crop_long_side=None, long_side=1024):
    """train --synthetic 하나를 돌리고 결과 CSV에 한 줄 기록한다.

    첫 step은 Triton 커널 최초 컴파일이 끼면 1~2분 멈춘 것처럼 보일 수 있다 —
    정상이다. 앞 3 step은 warmup으로 따로 두고, 이후 step들의 중앙값을
    steady-state sec/step으로 쓴다(평균은 컴파일 비용에 왜곡된다).
    """
    args = ["train", "-c", "9b_qlora_r8_1024", "--synthetic", "-n", str(n_steps),
            "--set", f"input.long_side={long_side}",
            "--set", f"input.crops={crops}",
            "--set", f"train.grad_checkpointing={'true' if grad_checkpointing else 'false'}"]
    if crop_long_side is not None:
        args += ["--set", f"input.crop_long_side={crop_long_side}"]

    row = {
        "hostname": HOSTNAME, "gpu_name": GPU_NAME, "mode": "train",
        "long_side": long_side, "crops": crops, "crop_long_side": crop_long_side,
    }
    print(f"=== [train] {name} ===")
    try:
        rc, output = _run_subprocess(args)
        status = _classify_status(rc, output)
        run_dir = REPO_ROOT / "runs" / "trainprobe_9b_qlora_r8_1024"
        summary_path = run_dir / "summary.json"
        if status == "ok" and summary_path.exists():
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            probe = summary.get("train_probe", {})
            row["peak_vram_torch"] = probe.get("peak_vram_gb_torch")
            row["peak_vram_smi"] = probe.get("peak_vram_gb_nvidia_smi")
            row["sec_per_step_steady"] = probe.get("steady_state_sec_per_step_median")
            row["sec_per_step_warmup"] = probe.get("warmup_sec_per_step")
            _apply_kernel_impl(row, probe.get("kernel_impl"))
        row["status"] = status
        print(f"  -> {status}  peak_vram_torch={row.get('peak_vram_torch')}  "
              f"steady sec/step={row.get('sec_per_step_steady')}  "
              f"warmup steps={row.get('sec_per_step_warmup')}")
        if status != "ok":
            print(output[-2000:])
    except subprocess.TimeoutExpired:
        row["status"] = "TIMEOUT"
        print("  -> TIMEOUT")
    except Exception as exc:
        row["status"] = f"ERROR({type(exc).__name__})"
        print("  -> ERROR:", exc)
    finally:
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
    _write_result_row(row)
    return row


## 5. 추론 VRAM sweep — 가벼운 것부터

Full 1024 → 1536 → 2048. 여기서부터 셧다운이 나면 해상도 축만으로도 문제라는 뜻이다.

In [ ]:
measure_infer("long_side=1024 (baseline)", long_side=1024, crops=0)


In [ ]:
measure_infer("long_side=1536", long_side=1536, crops=0)


In [ ]:
measure_infer("long_side=2048", long_side=2048, crops=0)


## 6. crop VRAM sweep

crop 축은 09/17 세션에서 16GB를 넘긴 첫 지점이었다(crops=4, crop_long_side 기본값=1024). crop_long_side를 낮춘 3개 조합을 본다.

In [ ]:
measure_infer("crops=4, crop_long_side=768", long_side=1024, crops=4, crop_long_side=768)


In [ ]:
measure_infer("crops=4, crop_long_side=512", long_side=1024, crops=4, crop_long_side=512)


In [ ]:
measure_infer("crops=9, crop_long_side=512", long_side=1024, crops=9, crop_long_side=512)


## 7. 학습 VRAM probe — 가장 무거움, 맨 마지막

**셧다운이 재현된다면 여기서 날 가능성이 가장 높다.** 12 step 돌린다(loss forward →
backward → optimizer step). grad_checkpointing on/off를 비교하고, crop 학습도 하나 본다.

**⚠️ 첫 step에서 1~2분 멈춘 것처럼 보일 수 있다 — 정상이다.** fla/triton이 설치돼
있으면 Gated DeltaNet 커널을 Triton이 그 자리에서 컴파일한다(입력 크기별로 최초
1회, 09/17 실측 약 76초). **멈춘 줄 알고 Ctrl+C를 누르지 말 것.** 앞 3 step은
이 컴파일 비용이 섞인 warmup으로 따로 보고하고, 이후 9 step의 **중앙값**을
steady-state sec/step으로 쓴다(평균이 아니다 — 컴파일 비용 하나가 평균을 크게 왜곡한다).

`grad_checkpointing=False`는 09/17 세션에서 backward 단계 OOM으로 확인됐다 — 여기서도
OOM으로 끝나는 게 정상이다(그 자체가 재현 확인이다). 노트북은 안 죽는다.


In [ ]:
measure_train("grad_checkpointing=ON, long_side=1024 (baseline)",
              grad_checkpointing=True, crops=0, long_side=1024)


In [ ]:
measure_train("grad_checkpointing=OFF, long_side=1024 (OOM 예상 — 정상)",
              grad_checkpointing=False, crops=0, long_side=1024)


In [ ]:
measure_train("crop 학습: crops=4, crop_long_side=768, grad_checkpointing=ON",
              grad_checkpointing=True, crops=4, crop_long_side=768, long_side=1024)


## 8. 마무리

전력 로거를 멈추고 두 CSV의 위치를 다시 출력한다. **셧다운으로 여기까지 못 왔어도
`power_log_*.csv`는 이미 디스크에 있다** — 노트북이 죽었어도 그 파일만 챙겨서 보내면 된다.


In [ ]:
_power_log_stop.set()
_power_log_thread.join(timeout=5)
print("전력 로거 정지.")
print("\n결과 파일 위치:")
print(" ", Path(POWER_LOG_PATH).resolve())
print(" ", Path(RESULTS_PATH).resolve())
print("\n이 두 파일을 보내주면 된다. 노트북이 중간에 멈췄어도 power_log 파일은 그 직전까지 남아 있다.")
